# Feature engineering

Run after `preprocessing.ipynb`. Features are grouped by city and lag features use only earlier AQI observations.


In [ ]:
from pathlib import Path
import pandas as pd

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data/interim/city_day_clean.csv").exists())
source = ROOT / "data/interim/city_day_clean.csv"
output = ROOT / "data/processed/city_day_features.csv"
df = pd.read_csv(source, parse_dates=["Date"]).sort_values(["City", "Date"]).reset_index(drop=True)


In [ ]:
df["Year"] = df["Date"].dt.year
df["Month"] = df["Date"].dt.month
df["Day"] = df["Date"].dt.day
df["DayOfWeek"] = df["Date"].dt.dayofweek
df["Season"] = df["Month"].map({12:"Winter", 1:"Winter", 2:"Winter", 3:"Summer", 4:"Summer", 5:"Summer", 6:"Monsoon", 7:"Monsoon", 8:"Monsoon", 9:"Monsoon", 10:"Post Monsoon", 11:"Post Monsoon"})
for lag in [1, 2, 3, 7, 30]:
    df[f"AQI_lag{lag}"] = df.groupby("City")["AQI"].shift(lag)
df["AQI_rolling7"] = df.groupby("City")["AQI"].transform(lambda values: values.shift(1).rolling(7, min_periods=1).mean())
features = df.dropna(subset=["AQI_lag1"]).copy()
output.parent.mkdir(parents=True, exist_ok=True)
features.to_csv(output, index=False)
print(f"Saved {len(features):,} feature rows to {output}")
features.head()
